In [1]:
import pandas as pd
import numpy as np

# NLP
import re
import nltk
from nltk.corpus import stopwords
from textblob import TextBlob

# ML
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\adity\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [2]:
df = pd.read_csv('archive/tweets/data_analysis.csv')

C:\Users\adity\AppData\Local\Temp\ipykernel_26688\622318955.py:1: DtypeWarning: Columns (22) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('archive/tweets/data_analysis.csv')


In [3]:
df.head()

,id,conversation_id,created_at,date,time,timezone,user_id,username,name,place,...,geo,source,user_rt_id,user_rt,retweet_id,reply_to,retweet_date,translate,trans_src,trans_dest
0,1406397735792791556,1406397735792791556,2021-06-20 05:15:24 IST,2021-06-20,05:15:24,530,16001827,spj_tweets,Society of Professional Journalists,NaN,...,NaN,NaN,NaN,NaN,NaN,[],NaN,NaN,NaN,NaN
1,1406057066058313733,1406057066058313733,2021-06-19 06:41:42 IST,2021-06-19,06:41:42,530,1067308099,coastguardph,Philippine Coast Guard,NaN,...,NaN,NaN,NaN,NaN,NaN,[],NaN,NaN,NaN,NaN
2,1405965100993986561,1405965100993986561,2021-06-19 00:36:16 IST,2021-06-19,00:36:16,530,167406704,yaelkohen,Yael Kohen,NaN,...,NaN,NaN,NaN,NaN,NaN,[],NaN,NaN,NaN,NaN
3,1405935463374991366,1405935461042958338,2021-06-18 22:38:30 IST,2021-06-18,22:38:30,530,29206490,ieeeusa,IEEE-USA,NaN,...,NaN,NaN,NaN,NaN,NaN,[],NaN,NaN,NaN,NaN
4,1405918849329991694,1405918849329991694,2021-06-18 21:32:29 IST,2021-06-18,21:32:29,530,31696962,dw_europe,DW Europe,NaN,...,NaN,NaN,NaN,NaN,NaN,[],NaN,NaN,NaN,NaN


In [4]:
# Keep only tweet column
df = df[['tweet']]

# Drop null values
df.dropna(inplace=True)

df.head()

,tweet
0,Join @SPJ_IC at 6 p.m. EDT Tuesday for the nex...
1,"READ: Resilience, hard work, and perseverance ..."
2,How will the shift to streaming impact movie-m...
3,(2/2) to gain the aptitude to find better solu...
4,Alexander Lukashenko is still in power in Bela...


### Text Cleaning Function

In [5]:
negation_words = {
    'no', 'not', 'nor', 'never', 'cannot',
    'dont', 'didnt', 'isnt', 'wasnt', 'shouldnt', 'wouldnt',
    'couldnt', 'wont', 'cant', 'aint'
}
stop_words = set(stopwords.words('english')) - negation_words

def clean_text(text):
    text = text.lower()  # Lowercase
    text = re.sub(r"n['’]t\b", " not", text)  # Expand n't contractions
    text = re.sub(r"http\S+", "", text)  # Remove URLs
    text = re.sub(r"@\w+", "", text)  # Remove mentions
    text = re.sub(r"#\w+", "", text)  # Remove hashtags
    text = re.sub(r"[^a-zA-Z\s]", "", text)  # Remove special chars

    words = text.split()
    words = [word for word in words if word not in stop_words]  # Remove stop words except negations

    return " ".join(words)

df['clean_tweet'] = df['tweet'].apply(clean_text)

df.head()

,tweet,clean_tweet
0,Join @SPJ_IC at 6 p.m. EDT Tuesday for the nex...,join pm edt tuesday next created code behind p...
1,"READ: Resilience, hard work, and perseverance ...",read resilience hard work perseverance paid ge...
2,How will the shift to streaming impact movie-m...,shift streaming impact moviemaking interesting...
3,(2/2) to gain the aptitude to find better solu...,gain aptitude find better solutions realworld ...
4,Alexander Lukashenko is still in power in Bela...,alexander lukashenko still power belarus despi...


### Generate Sentiment Labels

In [6]:
def get_sentiment(text):
    polarity = TextBlob(text).sentiment.polarity
    return 'positive' if polarity > 0 else 'negative'

df['label'] = df['clean_tweet'].apply(get_sentiment)

df.head()

,tweet,clean_tweet,label
0,Join @SPJ_IC at 6 p.m. EDT Tuesday for the nex...,join pm edt tuesday next created code behind p...,negative
1,"READ: Resilience, hard work, and perseverance ...",read resilience hard work perseverance paid ge...,negative
2,How will the shift to streaming impact movie-m...,shift streaming impact moviemaking interesting...,positive
3,(2/2) to gain the aptitude to find better solu...,gain aptitude find better solutions realworld ...,positive
4,Alexander Lukashenko is still in power in Bela...,alexander lukashenko still power belarus despi...,negative


In [7]:
le = LabelEncoder()
df['label_encoded'] = le.fit_transform(df['label'])

df.head()

,tweet,clean_tweet,label,label_encoded
0,Join @SPJ_IC at 6 p.m. EDT Tuesday for the nex...,join pm edt tuesday next created code behind p...,negative,0
1,"READ: Resilience, hard work, and perseverance ...",read resilience hard work perseverance paid ge...,negative,0
2,How will the shift to streaming impact movie-m...,shift streaming impact moviemaking interesting...,positive,1
3,(2/2) to gain the aptitude to find better solu...,gain aptitude find better solutions realworld ...,positive,1
4,Alexander Lukashenko is still in power in Bela...,alexander lukashenko still power belarus despi...,negative,0


### Feature Extraction ( Text -> Numbers )

In [8]:
cv = CountVectorizer(max_features=5000, ngram_range=(1, 2))

X = cv.fit_transform(df['clean_tweet']).toarray()
y = df['label_encoded']

### Train - Test Split

In [9]:
Xtrain, Xtest, ytrain, ytest = train_test_split(X, y, test_size=0.2, random_state=42)

### Train Naive Bayes Model

In [10]:
model = MultinomialNB()
model.fit(Xtrain, ytrain)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None


### Predictions

In [11]:
ypred = model.predict(Xtest)

### Evaluation

In [12]:
print("Accuracy:", accuracy_score(ytest, ypred))
print("\nClassification Report:\n", classification_report(ytest, ypred))

Accuracy: 0.8622188139059305

Classification Report:
               precision    recall  f1-score   support

           0       0.89      0.85      0.87      4213
           1       0.83      0.88      0.85      3611

    accuracy                           0.86      7824
   macro avg       0.86      0.86      0.86      7824
weighted avg       0.86      0.86      0.86      7824



### Test custom input

In [13]:
def predict_sentiment(text):
    cleaned = clean_text(text)

    # Negation-aware override for common phrases that bag-of-words often misses
    if re.search(r"\b(not|no|never|dont|didnt|cant|wont|cannot)\s+(like|love|good|great|excellent|amazing)\b", cleaned):
        return "Negative 😡"
    if re.search(r"\b(not|no|never|dont|didnt|cant|wont|cannot)\s+(hate|bad|terrible|awful|worst)\b", cleaned):
        return "Positive 😊"

    vector = cv.transform([cleaned]).toarray()
    prediction = model.predict(vector)
    return "Positive 😊" if prediction[0] == 1 else "Negative 😡"

# Try your own sentence
print(predict_sentiment("i dont like this product"))

Negative 😡
